# Mutual Fund Portfolio Analysis

This notebook provides comprehensive analysis of your mutual fund portfolio including:
- Portfolio performance metrics
- Risk analysis
- Asset allocation visualization
- Correlation analysis
- Benchmark comparison

---

## 1. Import Required Libraries

Import necessary libraries including pandas, numpy, matplotlib, seaborn, and financial analysis libraries.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Import our custom modules
from mf_analyzer import MFPortfolioAnalyzer
from zerodha_integration import ZerodhaMFIntegration

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")
print(f"📅 Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. Load and Explore Portfolio Data

Load mutual fund portfolio data from Zerodha API or use sample data for demonstration.

In [ ]:
# Initialize the portfolio analyzer
analyzer = MFPortfolioAnalyzer()

# Option 1: Load sample data for demonstration
print("📊 Loading sample portfolio data...")
holdings_df = analyzer.load_sample_data()

# Display basic information about the dataset
print(f"\n📈 Portfolio Overview:")
print(f"Total number of funds: {len(holdings_df)}")
print(f"Data columns: {list(holdings_df.columns)}")
print(f"\n📋 Portfolio Holdings:")
display(holdings_df)

In [ ]:
# Option 2: Connect to Zerodha (uncomment to use real data)
# print("🔗 Connecting to Zerodha API...")
# zerodha = ZerodhaMFIntegration()
# 
# if zerodha.connect_zerodha():
#     real_holdings = zerodha.fetch_mf_holdings()
#     if real_holdings is not None:
#         holdings_df = real_holdings
#         print("✅ Using real Zerodha data!")
#         display(holdings_df)
# else:
#     print("⚠️ Using sample data instead")

## 3. Data Cleaning and Preprocessing

Handle missing values, convert data types, and prepare data for analysis.

In [ ]:
# Check for missing values
print("🔍 Checking data quality...")
print("\nMissing values:")
print(holdings_df.isnull().sum())

# Data types
print("\n📊 Data types:")
print(holdings_df.dtypes)

# Basic statistics
print("\n📈 Basic statistics:")
numeric_columns = ['current_value', 'invested_amount', 'absolute_return', 'return_percentage']
display(holdings_df[numeric_columns].describe())

# Clean any missing or invalid data
holdings_df = holdings_df.dropna()
print(f"\n✅ Data cleaned. Final dataset has {len(holdings_df)} records.")

## 4. Portfolio Performance Analysis

Calculate key performance metrics including returns, cumulative returns, and risk-adjusted metrics.

In [ ]:
# Get portfolio summary
summary = analyzer.get_portfolio_summary()
print("💰 Portfolio Performance Summary:")
print("=" * 40)
for key, value in summary.items():
    print(f"{key}: {value}")

# Calculate additional performance metrics
total_invested = holdings_df['invested_amount'].sum()
total_current = holdings_df['current_value'].sum()
total_return = total_current - total_invested
return_pct = (total_return / total_invested) * 100

print(f"\n📊 Additional Metrics:")
print(f"Portfolio Size: {len(holdings_df)} funds")
print(f"Average Fund Size: ₹{total_current/len(holdings_df):,.2f}")
print(f"Best Performer: {holdings_df.loc[holdings_df['return_percentage'].idxmax(), 'scheme_name']} ({holdings_df['return_percentage'].max():.2f}%)")
print(f"Worst Performer: {holdings_df.loc[holdings_df['return_percentage'].idxmin(), 'scheme_name']} ({holdings_df['return_percentage'].min():.2f}%)")

In [ ]:
# Top and bottom performers
print("🏆 Top 3 Performers:")
top_performers = analyzer.get_top_performers(3)
display(top_performers)

print("\n⚠️ Bottom 3 Performers:")
bottom_performers = analyzer.get_underperformers(3)
display(bottom_performers)

## 5. Risk Analysis and Metrics

Calculate volatility, risk metrics, and assess portfolio risk profile.

In [ ]:
# Risk analysis
returns = holdings_df['return_percentage']

# Calculate risk metrics
portfolio_volatility = returns.std()
portfolio_mean_return = returns.mean()
sharpe_ratio = portfolio_mean_return / portfolio_volatility if portfolio_volatility != 0 else 0

# Risk distribution
positive_returns = returns[returns > 0]
negative_returns = returns[returns < 0]

print("⚠️ Risk Analysis:")
print("=" * 30)
print(f"Portfolio Volatility: {portfolio_volatility:.2f}%")
print(f"Average Return: {portfolio_mean_return:.2f}%")
print(f"Risk-Adjusted Return (Sharpe-like): {sharpe_ratio:.3f}")
print(f"Funds with Positive Returns: {len(positive_returns)}/{len(returns)} ({len(positive_returns)/len(returns)*100:.1f}%)")
print(f"Funds with Negative Returns: {len(negative_returns)}/{len(returns)} ({len(negative_returns)/len(returns)*100:.1f}%)")

# Value at Risk (simplified)
var_95 = np.percentile(returns, 5)  # 5th percentile
print(f"Value at Risk (95%): {var_95:.2f}%")

In [ ]:
# Risk visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Return distribution
axes[0,0].hist(returns, bins=10, alpha=0.7, edgecolor='black')
axes[0,0].axvline(portfolio_mean_return, color='red', linestyle='--', label=f'Mean: {portfolio_mean_return:.2f}%')
axes[0,0].axvline(var_95, color='orange', linestyle='--', label=f'VaR 95%: {var_95:.2f}%')
axes[0,0].set_title('Return Distribution')
axes[0,0].set_xlabel('Return %')
axes[0,0].set_ylabel('Frequency')
axes[0,0].legend()

# Risk-Return scatter
fund_sizes = holdings_df['current_value'] / 1000  # Size in thousands
scatter = axes[0,1].scatter(portfolio_volatility, portfolio_mean_return, 
                           s=fund_sizes, alpha=0.6, c=returns, cmap='RdYlGn')
axes[0,1].set_title('Risk-Return Profile')
axes[0,1].set_xlabel('Risk (Volatility %)')
axes[0,1].set_ylabel('Return %')
plt.colorbar(scatter, ax=axes[0,1], label='Return %')

# Fund performance ranking
fund_names = holdings_df['scheme_name'].str[:15]  # Truncate names
colors = ['green' if r > 0 else 'red' for r in returns]
axes[1,0].barh(range(len(fund_names)), returns, color=colors, alpha=0.7)
axes[1,0].set_yticks(range(len(fund_names)))
axes[1,0].set_yticklabels(fund_names)
axes[1,0].set_title('Fund Performance Ranking')
axes[1,0].set_xlabel('Return %')

# Cumulative return curve (simulated)
cumulative_returns = (1 + returns/100).cumprod()
axes[1,1].plot(cumulative_returns.values, marker='o')
axes[1,1].set_title('Cumulative Returns (Simulated)')
axes[1,1].set_xlabel('Fund Index')
axes[1,1].set_ylabel('Cumulative Return')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Asset Allocation Visualization

Create pie charts and bar plots to visualize the current asset allocation across different mutual fund categories.

In [ ]:
# Asset allocation analysis
allocation = analyzer.analyze_allocation()
print("🎯 Asset Allocation by Category:")
display(allocation)

# Create comprehensive allocation visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Allocation by category (pie chart)
category_allocation = holdings_df.groupby('category')['current_value'].sum()
axes[0,0].pie(category_allocation.values, labels=category_allocation.index, 
              autopct='%1.1f%%', startangle=90)
axes[0,0].set_title('Portfolio Allocation by Category')

# 2. Allocation by value (bar chart)
axes[0,1].bar(category_allocation.index, category_allocation.values/1000, 
              color=['skyblue', 'lightcoral', 'lightgreen', 'gold', 'plum'])
axes[0,1].set_title('Category Allocation (₹000s)')
axes[0,1].set_ylabel('Value (₹000s)')
axes[0,1].tick_params(axis='x', rotation=45)

# 3. Top funds by value
top_5_funds = holdings_df.nlargest(5, 'current_value')
fund_labels = [name[:20] + '...' if len(name) > 20 else name for name in top_5_funds['scheme_name']]
axes[0,2].pie(top_5_funds['current_value'], labels=fund_labels, autopct='%1.1f%%')
axes[0,2].set_title('Top 5 Funds by Value')

# 4. Investment vs Current Value by category
cat_invested = holdings_df.groupby('category')['invested_amount'].sum()
cat_current = holdings_df.groupby('category')['current_value'].sum()

x = np.arange(len(cat_invested))
width = 0.35
axes[1,0].bar(x - width/2, cat_invested/1000, width, label='Invested', alpha=0.8)
axes[1,0].bar(x + width/2, cat_current/1000, width, label='Current', alpha=0.8)
axes[1,0].set_xlabel('Category')
axes[1,0].set_ylabel('Value (₹000s)')
axes[1,0].set_title('Invested vs Current Value by Category')
axes[1,0].set_xticks(x)
axes[1,0].set_xticklabels(cat_invested.index, rotation=45)
axes[1,0].legend()

# 5. Return percentage by category
cat_returns = holdings_df.groupby('category')['return_percentage'].mean()
colors = ['green' if r > 0 else 'red' for r in cat_returns]
axes[1,1].bar(cat_returns.index, cat_returns.values, color=colors, alpha=0.7)
axes[1,1].set_title('Average Returns by Category')
axes[1,1].set_ylabel('Return %')
axes[1,1].tick_params(axis='x', rotation=45)
axes[1,1].axhline(y=0, color='black', linestyle='-', alpha=0.3)

# 6. Fund count by category
fund_counts = holdings_df['category'].value_counts()
axes[1,2].bar(fund_counts.index, fund_counts.values, color='lightblue', alpha=0.8)
axes[1,2].set_title('Number of Funds by Category')
axes[1,2].set_ylabel('Number of Funds')
axes[1,2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 7. Correlation Analysis

Analyze correlations between different mutual funds in the portfolio using correlation matrices.

In [ ]:
# Create a simulated correlation matrix for demonstration
# In real scenario, this would use historical NAV data

print("📊 Correlation Analysis")
print("Note: Using simulated correlation data for demonstration")
print("In practice, this would use historical NAV data from each fund")

# Generate simulated correlation matrix
np.random.seed(42)  # for reproducible results
n_funds = len(holdings_df)
fund_names = [name[:15] for name in holdings_df['scheme_name']]  # Shortened names

# Create correlation matrix with realistic values
# Same category funds tend to be more correlated
base_corr = np.random.uniform(0.3, 0.8, (n_funds, n_funds))
correlation_matrix = (base_corr + base_corr.T) / 2  # Make symmetric
np.fill_diagonal(correlation_matrix, 1.0)  # Perfect self-correlation

# Create DataFrame for better visualization
corr_df = pd.DataFrame(correlation_matrix, 
                       index=fund_names, 
                       columns=fund_names)

print("\n🔍 Correlation Matrix:")
display(corr_df.round(3))

In [ ]:
# Visualize correlation matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap
sns.heatmap(corr_df, annot=True, cmap='coolwarm', center=0, 
            square=True, ax=axes[0], fmt='.2f')
axes[0].set_title('Fund Correlation Heatmap')
axes[0].set_xlabel('Funds')
axes[0].set_ylabel('Funds')

# Correlation distribution
# Get upper triangle of correlation matrix (excluding diagonal)
mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)
corr_values = corr_df.where(mask).stack().values

axes[1].hist(corr_values, bins=15, alpha=0.7, edgecolor='black')
axes[1].axvline(np.mean(corr_values), color='red', linestyle='--', 
                label=f'Mean: {np.mean(corr_values):.3f}')
axes[1].set_title('Distribution of Fund Correlations')
axes[1].set_xlabel('Correlation Coefficient')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

# Correlation insights
avg_correlation = np.mean(corr_values)
max_correlation = np.max(corr_values)
min_correlation = np.min(corr_values)

print(f"\n📈 Correlation Insights:")
print(f"Average Correlation: {avg_correlation:.3f}")
print(f"Highest Correlation: {max_correlation:.3f}")
print(f"Lowest Correlation: {min_correlation:.3f}")

if avg_correlation > 0.7:
    print("⚠️ High correlation detected - consider diversifying across different categories")
elif avg_correlation < 0.3:
    print("✅ Good diversification - low correlation between funds")
else:
    print("💡 Moderate correlation - reasonable diversification")

## 8. Performance Comparison with Benchmarks

Compare portfolio performance against relevant market benchmarks and indices.

In [ ]:
# Benchmark comparison
# For demonstration, we'll use typical benchmark returns

# Common benchmark returns (annual) - these would be fetched from real data
benchmarks = {
    'NIFTY 50': 12.5,
    'NIFTY 500': 13.2,
    'BSE Sensex': 12.1,
    'NIFTY Small Cap': 15.8,
    'NIFTY Mid Cap': 14.3
}

# Portfolio metrics
portfolio_return = holdings_df['return_percentage'].mean()
portfolio_total_return = ((holdings_df['current_value'].sum() - holdings_df['invested_amount'].sum()) 
                         / holdings_df['invested_amount'].sum()) * 100

print("📊 Benchmark Comparison:")
print("=" * 40)
print(f"Portfolio Average Return: {portfolio_return:.2f}%")
print(f"Portfolio Total Return: {portfolio_total_return:.2f}%")
print("\nBenchmark Returns:")
for benchmark, return_val in benchmarks.items():
    outperformance = portfolio_return - return_val
    status = "✅ Outperforming" if outperformance > 0 else "❌ Underperforming"
    print(f"{benchmark}: {return_val:.2f}% | {status} by {abs(outperformance):.2f}%")

In [ ]:
# Visualize benchmark comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Portfolio vs Benchmarks bar chart
all_returns = list(benchmarks.values()) + [portfolio_return]
all_labels = list(benchmarks.keys()) + ['Your Portfolio']
colors = ['lightblue'] * len(benchmarks) + ['orange']

axes[0,0].bar(all_labels, all_returns, color=colors, alpha=0.8)
axes[0,0].set_title('Portfolio vs Benchmark Returns')
axes[0,0].set_ylabel('Return %')
axes[0,0].tick_params(axis='x', rotation=45)
axes[0,0].axhline(y=portfolio_return, color='red', linestyle='--', alpha=0.7)

# 2. Category-wise performance vs relevant benchmarks
category_returns = holdings_df.groupby('category')['return_percentage'].mean()
axes[0,1].bar(category_returns.index, category_returns.values, 
              color=['green' if r > 12 else 'orange' if r > 8 else 'red' for r in category_returns])
axes[0,1].set_title('Category Performance vs Market Average (12%)')
axes[0,1].set_ylabel('Return %')
axes[0,1].axhline(y=12, color='black', linestyle='--', label='Market Average')
axes[0,1].tick_params(axis='x', rotation=45)
axes[0,1].legend()

# 3. Risk-adjusted performance (Sharpe-like ratio)
portfolio_sharpe = portfolio_return / holdings_df['return_percentage'].std()
benchmark_sharpes = {name: ret/3 for name, ret in benchmarks.items()}  # Assuming 3% volatility

sharpe_names = list(benchmark_sharpes.keys()) + ['Your Portfolio']
sharpe_values = list(benchmark_sharpes.values()) + [portfolio_sharpe]
sharpe_colors = ['lightcoral'] * len(benchmark_sharpes) + ['darkgreen']

axes[1,0].bar(sharpe_names, sharpe_values, color=sharpe_colors, alpha=0.8)
axes[1,0].set_title('Risk-Adjusted Returns (Sharpe-like Ratio)')
axes[1,0].set_ylabel('Risk-Adjusted Return')
axes[1,0].tick_params(axis='x', rotation=45)

# 4. Cumulative performance comparison (simulated)
months = np.arange(1, 13)
portfolio_cumulative = (1 + portfolio_return/100/12) ** months
nifty_cumulative = (1 + benchmarks['NIFTY 50']/100/12) ** months

axes[1,1].plot(months, portfolio_cumulative, marker='o', label='Your Portfolio', linewidth=2)
axes[1,1].plot(months, nifty_cumulative, marker='s', label='NIFTY 50', linewidth=2)
axes[1,1].set_title('Cumulative Performance (12 months)')
axes[1,1].set_xlabel('Months')
axes[1,1].set_ylabel('Cumulative Return')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 🎯 Key Insights and Recommendations

Based on the analysis above, here are the key insights about your mutual fund portfolio:

In [ ]:
# Generate key insights and recommendations
total_invested = holdings_df['invested_amount'].sum()
total_current = holdings_df['current_value'].sum()
total_return = total_current - total_invested
return_pct = (total_return / total_invested) * 100

print("🎯 KEY INSIGHTS & RECOMMENDATIONS")
print("=" * 50)

# Portfolio Health Check
print("\n💰 PORTFOLIO HEALTH:")
if return_pct > 15:
    print("✅ Excellent performance! Your portfolio is outperforming most benchmarks.")
elif return_pct > 8:
    print("✅ Good performance. Portfolio is generating decent returns.")
elif return_pct > 0:
    print("⚠️ Moderate performance. Consider reviewing underperforming funds.")
else:
    print("❌ Portfolio is in losses. Immediate review and rebalancing recommended.")

# Diversification Analysis
print("\n🎯 DIVERSIFICATION:")
categories = holdings_df['category'].nunique()
if categories >= 4:
    print(f"✅ Well diversified across {categories} categories.")
elif categories >= 2:
    print(f"⚠️ Moderately diversified across {categories} categories. Consider adding more variety.")
else:
    print(f"❌ Poor diversification. Only {categories} category. High risk!")

# Risk Assessment
print("\n⚠️ RISK ASSESSMENT:")
negative_returns = len(holdings_df[holdings_df['return_percentage'] < 0])
if negative_returns == 0:
    print("✅ All funds are in profit. Low current risk.")
elif negative_returns <= len(holdings_df) * 0.3:
    print(f"⚠️ {negative_returns} fund(s) in loss. Moderate risk.")
else:
    print(f"❌ {negative_returns} fund(s) in loss. High risk portfolio!")

# Specific Recommendations
print("\n💡 RECOMMENDATIONS:")

# Top performer
best_fund = holdings_df.loc[holdings_df['return_percentage'].idxmax()]
print(f"🏆 Continue SIP in top performer: {best_fund['scheme_name'][:30]}... ({best_fund['return_percentage']:.1f}%)")

# Worst performer
worst_fund = holdings_df.loc[holdings_df['return_percentage'].idxmin()]
if worst_fund['return_percentage'] < -10:
    print(f"❌ Consider exiting: {worst_fund['scheme_name'][:30]}... ({worst_fund['return_percentage']:.1f}%)")
elif worst_fund['return_percentage'] < 0:
    print(f"⚠️ Monitor closely: {worst_fund['scheme_name'][:30]}... ({worst_fund['return_percentage']:.1f}%)")

# Allocation recommendations
large_cap_allocation = holdings_df[holdings_df['category'] == 'Large Cap']['current_value'].sum() / total_current * 100
if large_cap_allocation < 40:
    print(f"💡 Consider increasing Large Cap allocation (currently {large_cap_allocation:.1f}%)")

print("\n📊 EXPORT OPTIONS:")
print("• Run analyzer.export_analysis() to save detailed Excel report")
print("• Use zerodha.export_to_csv() to export raw data")
print("• Screenshots of charts can be saved for tracking")

## 📁 Export Analysis

Export the complete analysis to Excel file for future reference.

In [ ]:
# Export analysis to Excel
try:
    filename = f"mf_portfolio_analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
    analyzer.export_analysis(filename)
    print(f"✅ Analysis exported successfully to: {filename}")
    
    # Also create a summary CSV
    summary_filename = f"portfolio_summary_{datetime.now().strftime('%Y%m%d')}.csv"
    holdings_df.to_csv(summary_filename, index=False)
    print(f"✅ Portfolio data exported to: {summary_filename}")
    
except Exception as e:
    print(f"❌ Export failed: {str(e)}")
    print("💡 You can manually save the data using the dataframes above")

print("\n🎉 Analysis Complete!")
print("Thank you for using the MF Portfolio Analyzer.")